# Spectral Extraction

Three chapters of preparation come together here. Chapter 6 straightened the frame, Chapter 4
found where the source falls on it, and Chapter 7 removed the sky from underneath it. What
remains is a 2D frame in which the only significant signal is the object itself, spread over a
handful of rows around a known, curved path. **Extraction** is the step that collapses those rows
into a single number per dispersion column, a one-dimensional spectrum.

The naive version is a sum: pick a few rows either side of the trace and add them up. That is
`BoxcarExtract`, and it is not a strawman: it is fast, it makes almost no assumptions, and for
a bright source it is close to optimal. But it has two weaknesses. It weights a pixel in the
faint wings of the profile exactly as heavily as one at the peak, even though the wing pixel
carries far more noise per unit signal; and it captures a *fraction* of the source's light that
depends on how wide the profile happens to be, which varies with wavelength.

`HorneExtract` addresses both by weighting each pixel by how much signal it should contain and
how noisy it is.

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models
from specreduce.background import Background
from specreduce.extract import BoxcarExtract, HorneExtract
from specreduce.tilt_solution import TiltSolution
from specreduce.tracing import FitTrace

from common import TILT_SOLUTION_FILE, fit_tilt_solution, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape
rows = np.arange(ny)

The three steps before this one are exactly as the previous chapters left them, so they compress
into one cell: load the tilt solution and rectify (Chapter 6), trace (Chapter 4), subtract the sky
(Chapter 7).

The extraction operates on the **background-subtracted** frame, and it needs that frame to carry
its uncertainty. That works end to end because `TiltSolution.resample` propagates the variance
through rectification and `Background.sub_image()` propagates it through the subtraction.
`HorneExtract` further down this chapter would refuse to run without it.

In [ ]:
if not TILT_SOLUTION_FILE.exists():  # normally written by Chapter 6
    fit_tilt_solution(arcs).to_asdf(TILT_SOLUTION_FILE)

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

rectified = solution.resample(obj)

trace = FitTrace(
    rectified, bins=32, peak_method="gaussian", trace_model=models.Chebyshev1D(4)
)
background = Background.two_sided(rectified, trace, separation=60, width=20)
image = background.sub_image()

trace_y = np.asarray(trace.trace)
print(
    f"image {image.flux.shape}, {image.flux.unit}, "
    f"uncertainty: {type(image.uncertainty).__name__}"
)

## Boxcar extraction

`BoxcarExtract` sums a fixed number of rows centered on the trace. The `width` parameter is the
full height of that aperture in pixels, and because the aperture follows the trace it stays
centered on the source as it curves across the frame.

In [ ]:
boxcar = BoxcarExtract(image, trace, width=11)
spectrum = boxcar.spectrum

flux = spectrum.flux.value
error = np.sqrt(spectrum.uncertainty.array)

print(f"spectrum: {type(spectrum).__name__}, {flux.shape}, {spectrum.flux.unit}")
print(f"flux at column 512: {flux[512]:.0f} +/- {error[512]:.0f} e-")

The result is a `specutils` `Spectrum` with an uncertainty attached, ready for the wavelength
solution that the later chapters will supply. Note what the spectral axis is *not*: it is still
pixels. Extraction does not calibrate wavelength.

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 3.6), sharex="all", constrained_layout=True, height_ratios=[2, 1]
)

axs[0].plot(flux, lw=0.7)
axs[0].set_ylabel("Flux [e$^-$]")
axs[1].plot(flux / error, lw=0.7, c="C1")
axs[1].set(xlim=(0, nx - 1), xlabel="Dispersion [pix]", ylabel="SNR")

print(f"median SNR {np.median(flux / error):.0f}")

This is the same spectrum Chapter 1 showed as a crude eleven-row sum, but now the sky is gone and
every point carries an uncertainty. The broad shape is the combined response of star, grism and
detector; the dips are stellar and telluric absorption. Signal-to-noise peaks in the middle of
the range and falls off at both ends, where the grism throughput drops.

### How wide should the aperture be?

`width` is the one parameter that matters, and it is a genuine trade-off. Widening the aperture
collects more of the source's light, but each additional row contributes its own read noise and
sky-subtraction residual while adding progressively less signal. There is an optimum, and it is
easy to find empirically.

In [ ]:
widths = np.array([3, 5, 7, 9, 11, 13, 17, 21, 27, 35])
captured = np.zeros(widths.size)
snr = np.zeros(widths.size)

for i, width in enumerate(widths):
    s = BoxcarExtract(image, trace, width=width).spectrum
    f = s.flux.value
    e = np.sqrt(s.uncertainty.array)
    captured[i] = np.median(f)
    snr[i] = np.median(f / e)

reference = captured[-1]
print(f"{'width':>6}  {'median flux':>12}  {'% of widest':>12}  {'median SNR':>11}")
for width, f, s in zip(widths, captured, snr):
    print(f"{width:>6}  {f:>12.0f}  {100 * f / reference:>12.1f}  {s:>11.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)
ax.plot(widths, snr, "o-", lw=0.9, ms=4)
ax.axvline(widths[snr.argmax()], c="0.6", lw=0.8, ls="--")
ax.text(
    widths[snr.argmax()] + 0.7,
    snr.min(),
    f"optimum, width {widths[snr.argmax()]}",
    size=8,
    c="0.4",
)
ax.set(xlabel="Aperture width [pix]", ylabel="Median SNR");

The curve rises steeply, peaks between widths 11 and 13, and declines slowly. That location is not
arbitrary: the source profile has a FWHM of about 6 pixels, and for a Gaussian the
signal-to-noise of a summed aperture is maximized at roughly 1.4 FWHM, about 8 pixels, with a
broad, forgiving maximum. The measured optimum sits slightly wider because this profile has more
extended wings than a Gaussian.

Notice how flat the peak is. Anything from 9 to 17 is within one percent of the best SNR, so
there is no need to tune this carefully. Notice also that the optimum does **not** capture all
the light: a width-11 aperture collects about 95 percent of what a width-35 aperture does. That
is the correct trade: the missing four percent is in wing pixels whose noise outweighs their
signal.

### Aperture losses depend on wavelength

Throwing away four percent of the light would be harmless if it were the *same* four percent
everywhere. It is not. The width of the source profile changes across the detector, since seeing,
focus and atmospheric dispersion all vary with wavelength, so a fixed aperture captures a
varying fraction.

Resampling the frame onto trace-relative coordinates, as Chapter 4 did to validate the trace,
makes this measurable.

In [ ]:
offsets = np.linspace(-30, 30, 241)
stack = np.array(
    [np.interp(trace_y[i] + offsets, rows, image.flux.value[:, i]) for i in range(nx)]
).T

block = 128
block_starts = range(0, nx, block)
total = np.array([stack[:, x : x + block].sum() for x in block_starts])

print(f"{'columns':>12}" + "".join(f"{f'width {w}':>11}" for w in (5, 9, 11, 15)))
for x, t in zip(block_starts, total):
    line = f"{f'{x}-{x + block}':>12}"
    for width in (5, 9, 11, 15):
        inside = np.abs(offsets) <= width / 2
        line += f"{stack[inside, x : x + block].sum() / t:>11.4f}"
    print(line)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.8), constrained_layout=True)

for width in (5, 9, 11, 15):
    inside = np.abs(offsets) <= width / 2
    fraction = (
        np.array([stack[inside, x : x + block].sum() for x in block_starts]) / total
    )
    ax.plot(
        np.array(block_starts) + block / 2,
        fraction,
        "o-",
        lw=0.9,
        ms=3,
        label=f"width {width}",
    )
    print(
        f"width {width:2d}: capture {fraction.min():.4f} to {fraction.max():.4f}"
        f"  ->  {100 * (fraction.max() / fraction.min() - 1):.1f}% variation"
    )

ax.legend(frameon=False, ncols=4, fontsize=8)
ax.set(xlabel="Dispersion [pix]", ylabel="Fraction of flux\ninside aperture");

A width-5 aperture captures between 80 and 85 percent of the light depending on where you are
in the spectrum, a **6 percent** wavelength-dependent error, smooth and slowly varying, which
is exactly the shape of a real spectral feature. Widening to 11 pixels cuts that to 1.4 percent
and to 15 pixels to 1.0 percent.

Whether six percent matters depends on the measurement. The equivalent width of a narrow line
barely notices a loss that varies smoothly over hundreds of pixels. Anything that compares the
continuum across wavelength does, because an aperture-loss curve of this shape is
indistinguishable from a real spectral feature. That is the strongest practical argument for a
**wide** boxcar aperture even at some cost in SNR, or for an extraction method that does not have
a hard aperture edge at all.

## Optimal extraction

`HorneExtract` implements the algorithm of
[Horne (1986)](https://ui.adsabs.harvard.edu/abs/1986PASP...98..609H/abstract). Rather than
summing pixels equally, it weights each one by the **spatial profile**, the expected fraction of
the source's light in that row, divided by its **variance**. Pixels near the peak, which carry
most of the signal, dominate; pixels in the wings, which carry mostly noise, contribute little
but are not discarded outright.

Because the weights depend on the variance, the algorithm cannot run without one. That is why
Chapter 7's uncertainty propagation mattered: passing `background.sub_image()` supplies both the
flux and its variance in a single object.

In [ ]:
horne = HorneExtract(image, trace)
horne_spectrum = horne.spectrum

horne_flux = horne_spectrum.flux.value
horne_error = np.sqrt(horne_spectrum.uncertainty.array)

clean = slice(300, 700)
print(f"{'method':>22}  {'median flux':>12}  {'median sigma':>13}  {'median SNR':>11}")
for label, f, e in (
    ("boxcar, width 11", flux, error),
    ("Horne", horne_flux, horne_error),
):
    print(
        f"{label:>22}  {np.median(f[clean]):>12.0f}  {np.median(e[clean]):>13.1f}  "
        f"{np.median(f[clean] / e[clean]):>11.1f}"
    )

The two are level: a median signal-to-noise of 489 for the boxcar against 484 for Horne, with
Horne recovering slightly less flux.

That is the *right* answer, and it is worth saying why before checking it. Optimal extraction
earns its name in the regime where read noise and sky dominate: faint sources, where
down-weighting the noisy wings genuinely helps. This source is bright: at roughly 240 000 e⁻ per
column it is overwhelmingly photon-limited, and when the source's own shot noise dominates there
is very little for clever weighting to recover. Parity is what theory predicts here.

The signal-to-noise is level, but the fluxes are not identical. Plotting the two spectra together
shows the difference.

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 3.8), sharex="all", constrained_layout=True, height_ratios=[2, 1]
)

axs[0].plot(flux, lw=0.7, label="boxcar, width 11")
axs[0].plot(horne_flux, lw=0.7, label="Horne")
axs[0].legend(frameon=False, fontsize=8)
axs[0].set_ylabel("Flux [e$^-$]")

axs[1].axhline(1, c="0.6", lw=0.8, ls="--")
axs[1].plot(horne_flux / flux, lw=0.7, c="C2")
axs[1].set(
    xlim=(0, nx - 1),
    ylim=(0.7, 1.1),
    xlabel="Dispersion [pix]",
    ylabel="Horne / boxcar",
)

print(f"flux ratio: median {np.median((horne_flux / flux)[clean]):.3f}")

Horne returns about three percent less flux than the boxcar, and the ratio is not quite constant
across the spectrum. Some difference is expected and legitimate: the two methods weight the
profile differently, so they do not measure quite the same quantity, and a width-11 boxcar
captures about 95 percent of the light while Horne's Gaussian kernel captures a different
fraction that varies as the profile width does. Neither number is "the" flux. What matters for
most science is that whichever you choose, you apply it consistently.

## Spatial profile options

`HorneExtract` offers two ways to model the cross-dispersion profile, set by `spatial_profile`.

- `"gaussian"` (the default) fits a single Gaussian to the average profile, optionally with a
  polynomial background term controlled by `bkgrd_prof`. It is stable and cheap, but assumes the
  profile shape is the same at every wavelength.
- `"interpolated_profile"` samples the profile empirically in a number of dispersion bins
  (`n_bins_interpolated_profile`, default 10) and interpolates between them. It follows a profile
  that changes shape along the spectrum, and it makes no assumption of Gaussianity, at the cost
  of more parameters and no support for a background model.

Given that the profile FWHM really does vary here, the empirical option is the better fit in
principle.

In [ ]:
profiles = (
    ("gaussian", "gaussian"),
    ("interpolated, 10 bins", "interpolated_profile"),
    (
        "interpolated, 25 bins",
        {"name": "interpolated_profile", "n_bins_interpolated_profile": 25},
    ),
)

print(
    f"{'spatial_profile':>24}  {'median flux':>12}  {'median sigma':>13}  {'median SNR':>11}"
)
for label, profile in profiles:
    s = HorneExtract(image, trace, spatial_profile=profile).spectrum
    f = s.flux.value
    e = np.sqrt(s.uncertainty.array)
    print(
        f"{label:>24}  {np.median(f[clean]):>12.0f}  {np.median(e[clean]):>13.1f}  "
        f"{np.median(f[clean] / e[clean]):>11.1f}"
    )

The empirical profiles recover about 13 percent *more* flux than the Gaussian, at a median SNR
about three percent higher, 497 against 483. The reason is the one Chapter 7 measured: the
target has wings well beyond a Gaussian core, and a Gaussian kernel does not model them, whereas
the empirical profile follows whatever the data show. Ten and twenty-five bins agree to a fraction
of a percent, so the profile is not changing shape along the dispersion axis on this source. The price of the empirical option is noise: each bin's profile is measured rather than
fitted, and on a faint source that measurement noise goes straight into the weights. Reach for it
when the profile genuinely departs from the model and you have the signal to measure it.

## Summary

- Extraction collapses the background-subtracted 2D frame to a 1D `Spectrum` with an uncertainty,
  on a **pixel** axis; wavelength calibration comes later.
- `BoxcarExtract` sums a fixed aperture centered on the trace. Its `width` has a broad SNR optimum
  near **11 pixels** here, about 1.4 times the profile FWHM, and the peak is flat enough that
  anything from 9 to 17 is within one percent.
- The optimum does not capture all the light, and **the fraction captured varies with
  wavelength**: 6.1 percent across the spectrum at width 5, 1.4 percent at width 11. A loss curve
  of that shape is indistinguishable from a real spectral feature, so for work that compares the
  continuum across wavelength, prefer a wide aperture over the nominal SNR optimum.
- `HorneExtract` weights pixels by profile over variance, so it needs an input uncertainty,
  which `Background.sub_image()` supplies, and which `TiltSolution.resample` propagates through
  rectification.
- **On a bright, photon-limited source, Horne buys nothing over a boxcar**, at 484 against 489 in
  median SNR here. That is the expected answer: optimal extraction earns its keep in the
  background-limited regime, not this one.
- Boxcar and Horne do not measure quite the same quantity. Horne returns about three percent
  less flux here, and the ratio drifts with wavelength as the profile width does. Neither is "the"
  flux; pick one and apply it consistently.
- `spatial_profile="interpolated_profile"` follows the measured profile instead of fitting a
  Gaussian. On this source it recovers 13 percent more flux at a slightly higher SNR, because the
  target's wings extend well beyond a Gaussian core. It costs measurement noise in the weights, so
  reach for it when the profile departs from the model and the signal is there to measure it.

Next: [Line Identification](09-line-identification.ipynb), which starts the work of putting a
wavelength on the horizontal axis.